# LSTM with attention

An encoder-decoder LSTM compresses the whole input sequence into a single vector, which becomes a bottleneck as sequences get longer. Attention removes this bottleneck: at each output step, the decoder looks back at all encoder states and focuses on the relevant ones. This notebook trains the same model with and without additive (Bahdanau) attention on a synthetic task and visualizes where the decoder looks.

**Aprendizado Profundo para Reconhecimento Visual** — Ifes Campus Serra
Autor: Thiago M. Paixão

In [ ]:
# If True, test is running on Colab. Otherwise, test is assumed to be offline.
TEST_ON_COLAB = True
FOLDERNAME = 'Colab Notebooks/Aprendizado Profundo para Reconhecimento Visual/08_transformers'  # only used if TEST_ON_COLAB is True

assert not (FOLDERNAME is None and TEST_ON_COLAB), "FOLDERNAME has to be set if TEST_ON_COLAB is True"

In [ ]:
# Create dataset directory
import os

if TEST_ON_COLAB:
    # This mounts your Google Drive to the Colab VM.
    from google.colab import drive

    drive.mount('/content/drive')
    os.chdir(f'/content/drive/My Drive/{FOLDERNAME}')

## Artificial dataset

The task is to reverse a sequence of digits. The first output depends on the last input, so the model has to keep track of the entire sequence.

All sequences have the same length, so no padding is needed. The only special token is `sos_token`, and no end token is needed because the output length equals the input length. With variable lengths, we would need a padding token, masked attention scores and a loss that ignores padding.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

seq_len = 20  # try reducing this to 5 and see how the model without attention behaves
vocab_size = 10  # digits from 0 to 9
sos_token = vocab_size  # start-of-sequence token, used as the first decoder input


def make_batch(batch_size):
    x = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    y = x.flip(dims=[1])
    return x, y


x, y = make_batch(1)
print("Input: ", x[0].tolist())
print("Target:", y[0].tolist())

## Model

At each decoder step $t$, additive attention compares the decoder state $s_t$ with every encoder state $h_i$:

$$e_{t,i} = v^\top \tanh(W_s s_t + W_h h_i), \qquad \alpha_{t,i} = \text{softmax}_i(e_{t,i}), \qquad c_t = \sum_i \alpha_{t,i} h_i$$

The context vector $c_t$ is a different summary of the input for each output step. Without attention, the context is always the last encoder state.

Here $s_t$ is the decoder state before step $t$ is computed, and the decoder starts from the last encoder state.

In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.W_s = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W_h = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, s, encoder_states):
        """
        Inputs:
        - s: Decoder state, of shape (N, H)
        - encoder_states: Encoder states, of shape (N, T, H)

        Returns a tuple of:
        - context: Weighted sum of the encoder states, of shape (N, H)
        - weights: Attention weights, of shape (N, T)
        """
        query = self.W_s(s).unsqueeze(1)  # (N, H) -> (N, 1, H), broadcast over the T encoder states
        keys = self.W_h(encoder_states)  # (N, T, H)
        scores = self.v(torch.tanh(query + keys)).squeeze(-1)  # (N, T, 1) -> (N, T); dim=-1 keeps the batch dim when N=1
        weights = F.softmax(scores, dim=1)  # (N, T), each row sums to 1
        context = (weights.unsqueeze(-1) * encoder_states).sum(dim=1)  # (N, T, 1) * (N, T, H) -> (N, H)
        return context, weights


class Seq2Seq(nn.Module):
    def __init__(self, hidden_size=64, use_attention=True):
        super().__init__()
        self.use_attention = use_attention
        # Lookup table: one trainable vector per token (10 digits + SOS), shared by encoder and decoder
        self.embedding = nn.Embedding(vocab_size + 1, hidden_size)
        # nn.LSTM runs over the whole input sequence in a single call
        self.encoder = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        # nn.LSTMCell runs one step: each decoder input depends on the previous step.
        # Its input is the token embedding concatenated with the context, hence 2 * hidden_size
        self.decoder = nn.LSTMCell(2 * hidden_size, hidden_size)
        self.attention = AdditiveAttention(hidden_size)
        self.output = nn.Linear(2 * hidden_size, vocab_size)

    def forward(self, x, y=None):
        """
        If the target y is given, the decoder receives the correct previous token
        (teacher forcing). Otherwise, it receives its own previous prediction.

        Returns the logits, of shape (N, T, C), and the attention weights, of shape (N, T, T).
        """
        encoder_states, (s, c) = self.encoder(self.embedding(x))
        s, c = s[0], c[0]  # (num_layers, N, H) -> (N, H)
        last_state = s
        token = torch.full((x.size(0),), sos_token, device=x.device)

        logits, attention_weights = [], []
        for t in range(x.size(1)):
            if self.use_attention:
                context, weights = self.attention(s, encoder_states)
                attention_weights.append(weights)
            else:
                context = last_state
            s, c = self.decoder(torch.cat([self.embedding(token), context], dim=1), (s, c))
            step_logits = self.output(torch.cat([s, context], dim=1))
            logits.append(step_logits)
            token = y[:, t] if y is not None else step_logits.argmax(dim=1)

        weights = torch.stack(attention_weights, dim=1) if self.use_attention else None
        return torch.stack(logits, dim=1), weights

## Training

In [ ]:
def train(model, num_steps=1500, batch_size=128, learning_rate=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_history = []
    for step in range(num_steps):
        x, y = make_batch(batch_size)
        logits, _ = model(x, y)
        loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_history.append(loss.item())
        if (step + 1) % 300 == 0:
            print(f"Step {step + 1}/{num_steps}, Loss: {loss.item():.4f}")
    return loss_history


print("Without attention")
model_plain = Seq2Seq(use_attention=False).to(device)
history_plain = train(model_plain)

print("With attention")
model_attention = Seq2Seq(use_attention=True).to(device)
history_attention = train(model_attention)

In [ ]:
plt.plot(history_plain, label="Without attention")
plt.plot(history_attention, label="With attention")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss History")
plt.show()

## Test

At test time the decoder only sees its own predictions.

In [ ]:
@torch.no_grad()
def accuracy(model, batch_size=1000):
    x, y = make_batch(batch_size)
    logits, _ = model(x)
    predictions = logits.argmax(dim=2)
    return (predictions == y).float().mean().item(), (predictions == y).all(dim=1).float().mean().item()


for name, model in (("Without attention", model_plain), ("With attention", model_attention)):
    token_accuracy, sequence_accuracy = accuracy(model)
    print(f"{name}: token accuracy={token_accuracy:.3f}, sequence accuracy={sequence_accuracy:.3f}")

## Visualizing the attention

Each row shows where the decoder looked to produce one output. Since the task is to reverse the input, the weights should concentrate on the anti-diagonal: the first output looks at the last input, and so on.

In [ ]:
with torch.no_grad():
    x, y = make_batch(1)
    logits, weights = model_attention(x)

plt.imshow(weights[0].cpu(), cmap="viridis")
plt.xticks(range(seq_len), x[0].tolist())
plt.yticks(range(seq_len), logits[0].argmax(dim=1).tolist())
plt.xlabel("Input sequence")
plt.ylabel("Predicted sequence")
plt.title("Attention weights")
plt.colorbar()
plt.show()